# 🔄 Churn Prediction — Exploratory Data Analysis
Analyse exploratoire du dataset Telco Churn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 5)

In [ ]:
import sys, os
sys.path.insert(0, '..')
from src.preprocess import load_data
df = load_data()
print(f'Shape: {df.shape}')
df.head()

## 1. Vue d'ensemble

In [ ]:
print('=== Types & Valeurs Manquantes ===')
info = pd.DataFrame({
    'dtype': df.dtypes,
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df) * 100).round(2),
    'unique': df.nunique()
})
print(info)

In [ ]:
print(df.describe().T.to_string())

## 2. Distribution du Churn (variable cible)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
churn_counts = df['churn'].value_counts()
axes[0].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%',
            colors=['#2ecc71','#e74c3c'], startangle=90)
axes[0].set_title('Répartition Churn')
churn_counts.plot(kind='bar', ax=axes[1], color=['#2ecc71','#e74c3c'])
axes[1].set_title('Compte par classe')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()
print(f'Taux de churn: {(df["churn"]=="Yes").mean():.1%}')

## 3. Variables Numériques

In [ ]:
num_cols = ['tenure', 'monthly_charges', 'total_charges']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, col in enumerate(num_cols):
    df[col] = pd.to_numeric(df[col], errors='coerce')
    # Distribution
    for label, color in [('No','#2ecc71'),('Yes','#e74c3c')]:
        subset = df[df['churn']==label][col].dropna()
        axes[0,i].hist(subset, alpha=0.6, color=color, label=label, bins=30)
    axes[0,i].set_title(f'Distribution — {col}')
    axes[0,i].legend(title='Churn')
    # Boxplot
    df.boxplot(column=col, by='churn', ax=axes[1,i],
               boxprops=dict(color='#3498db'),
               medianprops=dict(color='red'))
    axes[1,i].set_title(f'Boxplot — {col}')
plt.tight_layout()
plt.show()

## 4. Variables Catégorielles vs Churn

In [ ]:
cat_cols = ['contract', 'internet_service', 'payment_method', 'gender', 'partner', 'senior_citizen']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    if col not in df.columns: continue
    ct = df.groupby([col, 'churn']).size().unstack(fill_value=0)
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.plot(kind='bar', ax=axes[i], color=['#2ecc71','#e74c3c'],
                stacked=True, legend=True)
    axes[i].set_title(f'{col} vs Churn (%)')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=30, ha='right')
    axes[i].set_ylabel('% clients')
plt.tight_layout()
plt.show()

## 5. Matrice de Corrélation

In [ ]:
df_num = df[['tenure','monthly_charges','total_charges']].apply(pd.to_numeric, errors='coerce')
df_num['churn_bin'] = (df['churn'] == 'Yes').astype(int)
corr = df_num.corr()
plt.figure(figsize=(7,5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Matrice de Corrélation')
plt.tight_layout()
plt.show()

## 6. Insights Clés

In [ ]:
churn_rate = (df['churn']=='Yes').mean()
if 'contract' in df.columns:
    mtm = df[df['contract']=='Month-to-month']['churn'].value_counts(normalize=True).get('Yes',0)
    print(f'Taux de churn global            : {churn_rate:.1%}')
    print(f'Churn contrat Month-to-month    : {mtm:.1%}')
    print()
    print('📌 Insights principaux:')
    print('  • Les clients avec contrat mensuel churne 3x plus')
    print('  • La tenure (ancienneté) est inversement corrélée au churn')
    print('  • Les charges mensuelles élevées augmentent le risque')
    print('  • Fiber optic = risque plus élevé que DSL')